In [9]:
# === Combine two skeletons (AXON + DENDRITE) into ONE SWC =====================
# - Axon comes from `axon_root_id` → labeled type=2
# - Dendrite comes from `dend_root_id` → labeled type=3
# - Only the dendrite has a soma. Provide soma on dendrite ONLY (soma_den_px or soma_den_nm).
# - The two trees are NOT connected in the SWC.
#
# Output SWC types: 2 = axon, 3 = dendrite (no type=1 node is added).
# ==============================================================================

from pathlib import Path
from typing import Sequence, Optional, Tuple
import numpy as np
import caveclient
import pcg_skel

# ---- Config (adjust if you like) ----
DATASTACK = "wclee_mouse_spinalcord_cltmr"
RES_NM_PER_PX = (32.0, 32.0, 45.0)        # (x,y,z) nm/px
SOMA_COLLAPSE_RADIUS_NM = 7500
DEFAULT_RADIUS_NM = 1000.0                 # in nm (converted if you output "um" or "px")
SWC_UNITS = "nm"                           # "nm" | "um" | "px"

# ---- Helpers ----------------------------------------------------------
def _px_to_nm(v_px: np.ndarray, res_nm_per_px: Sequence[float]) -> np.ndarray:
    return np.asarray(v_px, float) * np.asarray(res_nm_per_px, float)[None, :]

def _nm_to_um(v_nm: np.ndarray) -> np.ndarray:
    return np.asarray(v_nm, float) / 1000.0

def _choose_root_idx(verts_px: np.ndarray,
                     soma_nm: Optional[Sequence[float]],
                     res_nm_per_px: Sequence[float]) -> int:
    if soma_nm is None:
        return 0
    verts_nm = _px_to_nm(verts_px, res_nm_per_px)
    soma_nm = np.asarray(soma_nm, float).reshape(1, 3)
    d2 = np.sum((verts_nm - soma_nm) ** 2, axis=1)
    return int(np.argmin(d2))

def _bfs_parent(n_vertices: int, edges: np.ndarray, root_idx: int) -> np.ndarray:
    edges = np.asarray(edges, dtype=int)
    adj = [[] for _ in range(n_vertices)]
    for i, j in edges:
        adj[i].append(j); adj[j].append(i)
    parent = np.full(n_vertices, -2, dtype=int)
    parent[root_idx] = -1
    q = [root_idx]
    while q:
        u = q.pop(0)
        for v in adj[u]:
            if parent[v] == -2:
                parent[v] = u
                q.append(v)
    parent[parent == -2] = -1
    return parent

def _coords_in_units(verts_px: np.ndarray,
                     units: str,
                     res_nm_per_px: Sequence[float],
                     default_radius_nm: float) -> Tuple[np.ndarray, np.ndarray]:
    if units == "nm":
        coords = _px_to_nm(verts_px, res_nm_per_px)
        radii = np.full(len(verts_px), default_radius_nm, float)
    elif units == "um":
        coords = _nm_to_um(_px_to_nm(verts_px, res_nm_per_px))
        radii = np.full(len(verts_px), default_radius_nm / 1000.0, float)
    elif units == "px":
        coords = np.asarray(verts_px, float)
        avg_res = float(np.mean(np.asarray(res_nm_per_px, float)))
        radii = np.full(len(verts_px), default_radius_nm / avg_res, float)
    else:
        raise ValueError("units must be one of {'nm','um','px'}")
    return coords, radii

def _fetch_skeleton_dend(client, root_id: int,
                         soma_den_nm: Optional[Sequence[float]],
                         res_nm_per_px: Sequence[float]):
    """Dendrite: honor soma (collapse near soma)."""
    if soma_den_nm is not None:
        skel, mesh, _ = pcg_skel.pcg_skeleton(
            root_id,
            client,
            return_mesh=True,
            root_point=np.asarray(soma_den_nm, float),
            root_point_resolution=tuple(res_nm_per_px),
            collapse_soma=True,
            collapse_radius=int(SOMA_COLLAPSE_RADIUS_NM),
            return_l2dict=True,
        )
    else:
        skel = pcg_skel.pcg_skeleton(root_id, client, return_mesh=False)
    return np.asarray(skel.vertices, float), np.asarray(skel.edges, int)

def _fetch_skeleton_axon(client, root_id: int):
    """Axon: NO soma is provided; just fetch a skeleton."""
    skel = pcg_skel.pcg_skeleton(root_id, client, return_mesh=False)
    return np.asarray(skel.vertices, float), np.asarray(skel.edges, int)

def export_axon_dend_combined_swc(
    axon_root_id: int,
    dend_root_id: int,
    out_path: str,
    *,
    # soma is ONLY on dendrite:
    soma_den_px: Optional[Sequence[float]] = None,
    soma_den_nm: Optional[Sequence[float]] = None,
    datastack: str = DATASTACK,
    res_nm_per_px: Sequence[float] = RES_NM_PER_PX,
    units: str = SWC_UNITS,
    default_radius_nm: float = DEFAULT_RADIUS_NM,
) -> str:
    """
    Build axon skeleton from `axon_root_id` (type=2) and dendrite skeleton from `dend_root_id` (type=3),
    and write them into ONE SWC. Only the dendrite uses a soma (if provided).
    The two trees remain unconnected.

    soma_den_px: dendrite soma in voxels (x,y,z); converted via res_nm_per_px
    soma_den_nm: dendrite soma in nanometers (x,y,z). Provide either px OR nm.
    """
    client = caveclient.CAVEclient(datastack)

    # normalize dend soma to nm
    if soma_den_nm is not None:
        soma_nm = np.asarray(soma_den_nm, float)
    elif soma_den_px is not None:
        soma_nm = np.asarray(soma_den_px, float) * np.asarray(res_nm_per_px, float)
    else:
        soma_nm = None

    # fetch both skeletons
    verts_de_px, edges_de = _fetch_skeleton_dend(client, dend_root_id, soma_nm, res_nm_per_px)
    verts_ax_px, edges_ax = _fetch_skeleton_axon(client, axon_root_id)

    # choose roots (dendrite uses soma; axon uses first vertex)
    root_de = _choose_root_idx(verts_de_px, soma_nm, res_nm_per_px)
    root_ax = 0

    parent_de = _bfs_parent(len(verts_de_px), edges_de, root_de)
    parent_ax = _bfs_parent(len(verts_ax_px), edges_ax, root_ax)

    # convert to output units
    coords_de, radii_de = _coords_in_units(verts_de_px, units, res_nm_per_px, default_radius_nm)
    coords_ax, radii_ax = _coords_in_units(verts_ax_px, units, res_nm_per_px, default_radius_nm)

    # prepare rows: write AXON first (type=2), then DENDRITE (type=3)
    rows = []
    # axon block
    for i in range(len(coords_ax)):
        n_id = i + 1
        p_id = (int(parent_ax[i]) + 1) if parent_ax[i] >= 0 else -1
        x, y, z = coords_ax[i].tolist()
        rows.append((n_id, 2, x, y, z, float(radii_ax[i]), p_id))

    # dendrite block (offset indices)
    offset = len(coords_ax)
    for i in range(len(coords_de)):
        n_id = offset + i + 1
        p_id = (offset + int(parent_de[i]) + 1) if parent_de[i] >= 0 else -1
        x, y, z = coords_de[i].tolist()
        rows.append((n_id, 3, x, y, z, float(radii_de[i]), p_id))

    # write SWC
    out_path = str(out_path)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        f.write(
            "# Combined SWC (no auto split):\n"
            f"#  axon_root_id={axon_root_id} → type=2\n"
            f"#  dend_root_id={dend_root_id} → type=3 (soma on dendrite)\n"
            f"# Datastack: {datastack}\n"
            f"# Units: {units}\n"
            f"# Resolution (nm/px): {tuple(float(x) for x in res_nm_per_px)}\n"
            "# Columns: n type x y z radius parent\n"
        )
        for (n_id, t, x, y, z, r, p_id) in rows:
            f.write(f"{n_id} {t} {x:.3f} {y:.3f} {z:.3f} {r:.3f} {p_id}\n")

    print(f"[combined SWC] wrote {len(rows)} nodes: axon={len(coords_ax)} + dend={len(coords_de)} → {out_path}")
    return out_path

# -----------------------------
# Example (edit and run)
# -----------------------------
axon_root_id = 720575940900297186
dend_root_id = 720575940909284060
out_path = "/Users/wangchu/mesh_analysis/swc_by_tag/test_720575940937221662_preclean.swc"
soma_den_px = [24850, 2529, 2055]   # soma ONLY on dendrite; px→nm internally
export_axon_dend_combined_swc(axon_root_id, dend_root_id, out_path, soma_den_px=soma_den_px, units="nm")


[combined SWC] wrote 595 nodes: axon=325 + dend=270 → /Users/wangchu/mesh_analysis/swc_by_tag/test_720575940937221662_preclean.swc


'/Users/wangchu/mesh_analysis/swc_by_tag/test_720575940937221662_preclean.swc'

In [3]:
# === Export ONE skeleton as SWC (ALL NODES type=2 / "axon") ===================
# - You provide a single root_id.
# - (Optional) Provide soma (px or nm) to root/collapse near soma.
# - Every node is written with SWC type=2.
# ==============================================================================

from pathlib import Path
from typing import Sequence, Optional, Tuple
import numpy as np
import caveclient
import pcg_skel

# ---- Config (reuse/adjust) ----
DATASTACK = "wclee_mouse_spinalcord_cltmr"
RES_NM_PER_PX = (32.0, 32.0, 45.0)        # (x,y,z) nm/px
SOMA_COLLAPSE_RADIUS_NM = 7500
DEFAULT_RADIUS_NM = 1000.0                 # in nm (converted if you output "um" or "px")
SWC_UNITS = "nm"                           # "nm" | "um" | "px"

# ---- Helpers ----------------------------------------------------------
def _px_to_nm(v_px: np.ndarray, res_nm_per_px: Sequence[float]) -> np.ndarray:
    return np.asarray(v_px, float) * np.asarray(res_nm_per_px, float)[None, :]

def _nm_to_um(v_nm: np.ndarray) -> np.ndarray:
    return np.asarray(v_nm, float) / 1000.0

def _choose_root_idx(verts_px: np.ndarray,
                     soma_nm: Optional[Sequence[float]],
                     res_nm_per_px: Sequence[float]) -> int:
    if soma_nm is None:
        return 0
    verts_nm = _px_to_nm(verts_px, res_nm_per_px)
    soma_nm = np.asarray(soma_nm, float).reshape(1, 3)
    d2 = np.sum((verts_nm - soma_nm) ** 2, axis=1)
    return int(np.argmin(d2))

def _bfs_parent(n_vertices: int, edges: np.ndarray, root_idx: int) -> np.ndarray:
    edges = np.asarray(edges, dtype=int)
    adj = [[] for _ in range(n_vertices)]
    for i, j in edges:
        adj[i].append(j); adj[j].append(i)
    parent = np.full(n_vertices, -2, dtype=int)
    parent[root_idx] = -1
    q = [root_idx]
    while q:
        u = q.pop(0)
        for v in adj[u]:
            if parent[v] == -2:
                parent[v] = u
                q.append(v)
    parent[parent == -2] = -1
    return parent

def _coords_in_units(verts_px: np.ndarray,
                     units: str,
                     res_nm_per_px: Sequence[float],
                     default_radius_nm: float) -> Tuple[np.ndarray, np.ndarray]:
    if units == "nm":
        coords = _px_to_nm(verts_px, res_nm_per_px)
        radii = np.full(len(verts_px), default_radius_nm, float)
    elif units == "um":
        coords = _nm_to_um(_px_to_nm(verts_px, res_nm_per_px))
        radii = np.full(len(verts_px), default_radius_nm / 1000.0, float)
    elif units == "px":
        coords = np.asarray(verts_px, float)
        avg_res = float(np.mean(np.asarray(res_nm_per_px, float)))
        radii = np.full(len(verts_px), default_radius_nm / avg_res, float)
    else:
        raise ValueError("units must be one of {'nm','um','px'}")
    return coords, radii

def _fetch_skeleton(client, root_id: int,
                    soma_nm: Optional[Sequence[float]],
                    res_nm_per_px: Sequence[float]):
    """
    Fetch a skeleton. If soma is provided, root & collapse near soma.
    Returns vertices in voxels (px) and undirected edges (Nx2).
    """
    if soma_nm is not None:
        skel, mesh, _ = pcg_skel.pcg_skeleton(
            root_id,
            client,
            return_mesh=True,
            root_point=np.asarray(soma_nm, float),
            root_point_resolution=tuple(res_nm_per_px),
            collapse_soma=True,
            collapse_radius=int(SOMA_COLLAPSE_RADIUS_NM),
            return_l2dict=True,
        )
    else:
        skel = pcg_skel.pcg_skeleton(root_id, client, return_mesh=False)
    return np.asarray(skel.vertices, float), np.asarray(skel.edges, int)

def export_neuron_as_axon_swc(
    root_id: int,
    out_path: str,
    *,
    soma_px: Optional[Sequence[float]] = None,   # optional soma (x,y,z) in px
    soma_nm: Optional[Sequence[float]] = None,   # optional soma (x,y,z) in nm
    datastack: str = DATASTACK,
    res_nm_per_px: Sequence[float] = RES_NM_PER_PX,
    units: str = SWC_UNITS,
    default_radius_nm: float = DEFAULT_RADIUS_NM,
) -> str:
    """
    Export a single neuron skeleton as an SWC with ALL nodes labeled type=2 (axon).
    If soma is given, the skeleton will be rooted/collapsed near the soma.
    """
    client = caveclient.CAVEclient(datastack)

    # normalize soma to nm (if provided)
    if soma_nm is not None:
        soma_nm_arr = np.asarray(soma_nm, float)
    elif soma_px is not None:
        soma_nm_arr = np.asarray(soma_px, float) * np.asarray(res_nm_per_px, float)
    else:
        soma_nm_arr = None

    # fetch skeleton and pick root
    verts_px, edges = _fetch_skeleton(client, root_id, soma_nm_arr, res_nm_per_px)
    root_idx = _choose_root_idx(verts_px, soma_nm_arr, res_nm_per_px)
    parent = _bfs_parent(len(verts_px), edges, root_idx)

    # convert coordinates/radii to output units
    coords, radii = _coords_in_units(verts_px, units, res_nm_per_px, default_radius_nm)

    # write SWC (ALL type=2)
    out_path = str(out_path)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        f.write(
            "# Single-neuron SWC (all nodes type=2 / axon)\n"
            f"# root_id={root_id}\n"
            f"# Datastack: {datastack}\n"
            f"# Units: {units}\n"
            f"# Resolution (nm/px): {tuple(float(x) for x in res_nm_per_px)}\n"
            "# Columns: n type x y z radius parent\n"
        )
        for i in range(len(coords)):
            n_id = i + 1
            p_id = (int(parent[i]) + 1) if parent[i] >= 0 else -1
            x, y, z = coords[i].tolist()
            r = float(radii[i])
            f.write(f"{n_id} 2 {x:.3f} {y:.3f} {z:.3f} {r:.3f} {p_id}\n")

    print(f"[SWC] wrote {len(coords)} nodes (all type=2) → {out_path}")
    return out_path

# -----------------------------
# Example (edit and run)
# -----------------------------
# Your neuron root:
root_id = 720575940891163574
out_path = "/Users/wangchu/mesh_analysis/swc_by_tag/sensory_neuron/cltmr_720575940891163574_preclean.swc"

# Optional soma (choose one):
soma_px = [8836, 3004, 131]     # voxel coords
# soma_nm = [x_nm, y_nm, z_nm]  # or in nanometers

export_neuron_as_axon_swc(
    root_id,
    out_path,
    soma_px=soma_px,             # or soma_nm=...
    units="nm",                  # output in nm (or "px"/"um")
    res_nm_per_px=(32.0,32.0,45.0)  # set to (16,16,45) if that's your volume
)


[SWC] wrote 1967 nodes (all type=2) → /Users/wangchu/mesh_analysis/swc_by_tag/sensory_neuron/cltmr_720575940891163574_preclean.swc


'/Users/wangchu/mesh_analysis/swc_by_tag/sensory_neuron/cltmr_720575940891163574_preclean.swc'

In [10]:
# Remove the node with the largest X from an SWC (and detach its children),
# then save a new SWC. Parents pointing to the removed node are set to -1.
# We also (by default) RENNUMBER node IDs to keep them contiguous and
# remap all parent references accordingly.

from pathlib import Path
import re
import pandas as pd
import numpy as np

def load_swc_df(path):
    return pd.read_csv(
        path, comment="#", delim_whitespace=True, header=None,
        names=["n","type","x","y","z","radius","parent"],
        dtype={"n": int, "type": int, "x": float, "y": float, "z": float, "radius": float, "parent": int},
    )

def read_header_lines(path):
    hdr = []
    with open(path, "r") as f:
        while True:
            pos = f.tell()
            line = f.readline()
            if not line:
                break
            if line.startswith("#"):
                hdr.append(line.rstrip("\n"))
            else:
                break
    return hdr

def write_swc_with_header(df, header_lines, out_path, *, note=None):
    out_path = str(out_path)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        if header_lines:
            f.write("\n".join(header_lines) + "\n")
        if note:
            f.write(f"# {note}\n")
        # Ensure we have a columns line
        if not any(("Columns:" in h) for h in (header_lines or [])):
            f.write("# Columns: n type x y z radius parent\n")
        for _, r in df.iterrows():
            f.write(f"{int(r.n)} {int(r.type)} "
                    f"{float(r.x):.3f} {float(r.y):.3f} {float(r.z):.3f} "
                    f"{float(r.radius):.3f} {int(r.parent)}\n")

def clean_drop_max_x_node(swc_in, swc_out=None, *, renumber=True, print_info=True):
    """
    - Finds the row with the largest X coordinate.
    - Removes that node entirely.
    - Any children whose parent == removed_node_id get parent = -1.
    - Optionally renumbers remaining nodes to 1..N and remaps parent IDs.

    Returns (df_out, removed_record_dict, mapping_old_to_new or None)
    """
    swc_in = str(swc_in)
    if swc_out is None:
        swc_out = re.sub(r"\.swc$", "", swc_in) + ".dropmaxx.swc"

    df = load_swc_df(swc_in)
    hdr = read_header_lines(swc_in)

    if df.empty:
        raise ValueError("SWC file has no rows.")

    # 1) locate max-X node
    i_max = int(df["x"].idxmax())
    max_row = df.loc[i_max].copy()
    removed_id = int(max_row["n"])

    # 2) detach children (parent -> -1) for those pointing to the removed node
    child_mask = (df["parent"] == removed_id)
    num_children = int(child_mask.sum())
    df.loc[child_mask, "parent"] = -1

    # 3) drop the node row
    df = df.drop(index=i_max).reset_index(drop=True)

    mapping = None
    if renumber:
        # Build mapping old id -> new id (contiguous, in current row order)
        old_ids = df["n"].to_numpy(int)
        new_ids = np.arange(1, len(df) + 1, dtype=int)
        mapping = dict(zip(old_ids, new_ids))
        # apply mapping
        df["n"] = df["n"].map(mapping)
        # parent: -1 stays -1, others map via dict
        df["parent"] = df["parent"].apply(lambda p: -1 if int(p) == -1 else mapping.get(int(p), -1))

    note = (f"drop max-X node: old_id={removed_id}, type={int(max_row['type'])}, "
            f"x={float(max_row['x']):.3f}, y={float(max_row['y']):.3f}, z={float(max_row['z']):.3f}; "
            f"children_detached={num_children}; renumber={renumber}")

    write_swc_with_header(df, hdr, swc_out, note=note)

    if print_info:
        print("[clean-drop-max-x]")
        print("  input :", swc_in)
        print("  output:", swc_out)
        print("  removed node:", {
            "n": removed_id, "type": int(max_row["type"]),
            "x": float(max_row["x"]), "y": float(max_row["y"]), "z": float(max_row["z"]),
            "radius": float(max_row["radius"]), "parent": int(max_row["parent"])
        })
        print(f"  children detached: {num_children}")
        if renumber:
            print("  renumbered nodes to 1..N")

        # Show new max-X row for sanity
        i_new_max = int(df["x"].idxmax())
        print("  new max-X row:", df.loc[i_new_max, ["n","type","x","y","z","radius","parent"]].to_dict())

    removed_record = {
        "n": removed_id, "type": int(max_row["type"]),
        "x": float(max_row["x"]), "y": float(max_row["y"]), "z": float(max_row["z"]),
        "radius": float(max_row["radius"]), "parent": int(max_row["parent"]),
        "children_detached": num_children
    }
    return df, removed_record, mapping

# -----------------------------
# Run on your file
# -----------------------------
swc_in = "/Users/wangchu/mesh_analysis/swc_by_tag/test_720575940937221662_preclean.swc"
swc_out = "/Users/wangchu/mesh_analysis/swc_by_tag/test_720575940937221662.swc"

_ = clean_drop_max_x_node(swc_in, swc_out=swc_out, renumber=True, print_info=True)

# If you want to immediately print the new max nodes (optional):
df_new = load_swc_df(swc_out)
for col in ("x","y","z"):
    i = df_new[col].idxmax()
    print(f"Max-{col.upper()} after drop:", df_new.loc[i, ["n","type","x","y","z","radius","parent"]].to_dict())


[clean-drop-max-x]
  input : /Users/wangchu/mesh_analysis/swc_by_tag/test_720575940937221662_preclean.swc
  output: /Users/wangchu/mesh_analysis/swc_by_tag/test_720575940937221662.swc
  removed node: {'n': 595, 'type': 3, 'x': 814284800.0, 'y': 82870272.0, 'z': 187261875.0, 'radius': 1000.0, 'parent': 594}
  children detached: 0
  renumbered nodes to 1..N
  new max-X row: {'n': 324.0, 'type': 2.0, 'x': 33464320.0, 'y': 9677824.0, 'z': 2322675.0, 'radius': 1000.0, 'parent': 325.0}
Max-X after drop: {'n': 324.0, 'type': 2.0, 'x': 33464320.0, 'y': 9677824.0, 'z': 2322675.0, 'radius': 1000.0, 'parent': 325.0}
Max-Y after drop: {'n': 161.0, 'type': 2.0, 'x': 28215296.0, 'y': 11403264.0, 'z': 299700.0, 'radius': 1000.0, 'parent': 162.0}
Max-Z after drop: {'n': 22.0, 'type': 2.0, 'x': 23076864.0, 'y': 2656256.0, 'z': 6542775.0, 'radius': 1000.0, 'parent': 16.0}


/var/folders/xx/lz1tb24d0y306cdbdfblscxm0000gn/T/ipykernel_1015/3532060077.py:12: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  return pd.read_csv(
/var/folders/xx/lz1tb24d0y306cdbdfblscxm0000gn/T/ipykernel_1015/3532060077.py:12: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  return pd.read_csv(
